# Heaviest / Most-Held Stocks
This notebook loads the raw 13F Hedge Fund data from StockFries, filters for the most recent reporting period, and visualizes the "heaviest" aggregated positions (most owned across tracked hedge funds).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load dataset (adjust path if running on Kaggle)
# e.g., df = pd.read_csv('/kaggle/input/hedge-fund-13f-positions-from-stockfries-2018-2026/data_by_stocks.csv', low_memory=False)
df = pd.read_csv('../data_by_stocks.zip', low_memory=False)
df.head()

In [ ]:
# Ensure reportDate is datetime
df['reportDate'] = pd.to_datetime(df['reportDate'])

# Find most recent reporting period
most_recent_period = df['reportDate'].max()
print(f"Most recent reporting period: {most_recent_period.strftime('%Y-%m-%d')}")

# Filter data
df_recent = df[df['reportDate'] == most_recent_period]
print(f"Total positions held in the most recent period: {len(df_recent)}")

In [ ]:
# Aggregate to find Heaviest positions
heaviest_agg = df_recent.groupby('nameOfIssuer')['value'].sum()

# Convert to Millions ($ MMs) and sort
heaviest_agg = (heaviest_agg / 1000).sort_values(ascending=False).round(1)

top_20_heaviest = heaviest_agg.head(20)
top_20_df = top_20_heaviest.reset_index()
top_20_df.columns = ['nameOfIssuer', 'value_MMs']
display(top_20_df)

In [ ]:
plt.figure(figsize=(12, 8))
barplot = sns.barplot(x='value_MMs', y='nameOfIssuer', data=top_20_df, palette='viridis')

plt.title(f'Top 20 Heaviest Hedge Fund Holdings ({most_recent_period.strftime("%b %Y")})', fontsize=16, pad=15)
plt.xlabel('Aggregated Value ($ MMs)', fontsize=12)
plt.ylabel('Stock / Issuer', fontsize=12)

for i, p in enumerate(barplot.patches):
    width = p.get_width()
    plt.text(width + (width * 0.02), p.get_y() + p.get_height() / 2, f'${width:,.1f}M', 
             ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()